In [1]:
import requests
import time
import json
import csv
from pathlib import Path
from datetime import date, timedelta
import pandas as pd

In [2]:
START_DATE = date(2026, 1, 1)
END_DATE   = date(2026, 5, 12)  # today
OUTPUT     = "data_processed/spx_daily_history.csv"
DELAY      = 1  # seconds between requests

def date_to_slug(d):
    # e.g. spx-up-or-down-on-may-12-2026
    return f"spx-up-or-down-on-{d.strftime('%B').lower()}-{d.day}-{d.year}"

all_dates = [START_DATE + timedelta(days=i)
             for i in range((END_DATE - START_DATE).days + 1)]
slugs = [(d, date_to_slug(d)) for d in all_dates]

print(f"{len(slugs)} calendar days to probe")
print(f"ETA: ~{len(slugs) * DELAY // 60} min {len(slugs) * DELAY % 60} sec")
print(f"First slug: {slugs[0][1]}")
print(f"Last  slug: {slugs[-1][1]}")

132 calendar days to probe
ETA: ~2 min 12 sec
First slug: spx-up-or-down-on-january-1-2026
Last  slug: spx-up-or-down-on-may-12-2026


In [3]:
Path(OUTPUT).parent.mkdir(exist_ok=True)

with open(OUTPUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["date", "slug", "startDate", "endDate", "p_up", "p_down", "resolved", "outcome"])

    for i, (d, slug) in enumerate(slugs):
        resp = requests.get(
            "https://gamma-api.polymarket.com/events",
            params={"slug": slug}
        )
        data = resp.json()

        if not data:
            print(f"[{i+1:03d}/{len(slugs)}] {slug} — not found (weekend/holiday)")
        else:
            market = data[0]["markets"][0]
            prices = json.loads(market["outcomePrices"])
            p_up   = float(prices[0])
            p_down = float(prices[1])
            resolved = market.get("resolved", False)
            # infer outcome from settled prices (1.0 = winner)
            if resolved:
                outcome = "Up" if p_up > 0.5 else "Down"
            else:
                outcome = ""
            writer.writerow([
                d.isoformat(),
                market["slug"],
                market["startDate"],
                market["endDate"],
                p_up,
                p_down,
                resolved,
                outcome,
            ])
            f.flush()
            print(f"[{i+1:03d}/{len(slugs)}] {slug} — Up={p_up:.4f}  Down={p_down:.4f}  resolved={resolved}")

        if i < len(slugs) - 1:
            time.sleep(DELAY)

print(f"\nDone. Saved to {OUTPUT}")

[001/132] spx-up-or-down-on-january-1-2026 — not found (weekend/holiday)
[002/132] spx-up-or-down-on-january-2-2026 — Up=1.0000  Down=0.0000  resolved=False
[003/132] spx-up-or-down-on-january-3-2026 — not found (weekend/holiday)
[004/132] spx-up-or-down-on-january-4-2026 — not found (weekend/holiday)
[005/132] spx-up-or-down-on-january-5-2026 — Up=1.0000  Down=0.0000  resolved=False
[006/132] spx-up-or-down-on-january-6-2026 — Up=1.0000  Down=0.0000  resolved=False
[007/132] spx-up-or-down-on-january-7-2026 — Up=0.0000  Down=1.0000  resolved=False
[008/132] spx-up-or-down-on-january-8-2026 — Up=1.0000  Down=0.0000  resolved=False
[009/132] spx-up-or-down-on-january-9-2026 — Up=1.0000  Down=0.0000  resolved=False
[010/132] spx-up-or-down-on-january-10-2026 — not found (weekend/holiday)
[011/132] spx-up-or-down-on-january-11-2026 — not found (weekend/holiday)
[012/132] spx-up-or-down-on-january-12-2026 — Up=1.0000  Down=0.0000  resolved=False
[013/132] spx-up-or-down-on-january-13-2026 

In [4]:
df = pd.read_csv(OUTPUT)
df["date"]      = pd.to_datetime(df["date"])
df["startDate"] = pd.to_datetime(df["startDate"])
df["endDate"]   = pd.to_datetime(df["endDate"])

print(f"Total trading days fetched: {len(df)}")
df.head(10)

Total trading days fetched: 89


,date,slug,startDate,endDate,p_up,p_down,resolved,outcome
0,2026-01-02,spx-up-or-down-on-january-2-2026,2025-12-31 13:01:06.160002+00:00,2026-01-02 21:00:00+00:00,1.0,0.0,False,NaN
1,2026-01-05,spx-up-or-down-on-january-5-2026,2026-01-02 13:01:07.473463+00:00,2026-01-05 21:00:00+00:00,1.0,0.0,False,NaN
2,2026-01-06,spx-up-or-down-on-january-6-2026,2026-01-05 13:01:12.662022+00:00,2026-01-06 21:00:00+00:00,1.0,0.0,False,NaN
3,2026-01-07,spx-up-or-down-on-january-7-2026,2026-01-06 13:01:12.734748+00:00,2026-01-07 21:00:00+00:00,0.0,1.0,False,NaN
4,2026-01-08,spx-up-or-down-on-january-8-2026,2026-01-07 13:01:03.867669+00:00,2026-01-08 21:00:00+00:00,1.0,0.0,False,NaN
5,2026-01-09,spx-up-or-down-on-january-9-2026,2026-01-08 13:01:01.088510+00:00,2026-01-09 21:00:00+00:00,1.0,0.0,False,NaN
6,2026-01-12,spx-up-or-down-on-january-12-2026,2026-01-09 13:01:14.331037+00:00,2026-01-12 21:00:00+00:00,1.0,0.0,False,NaN
7,2026-01-13,spx-up-or-down-on-january-13-2026,2026-01-12 13:01:03.878223+00:00,2026-01-13 21:00:00+00:00,0.0,1.0,False,NaN
8,2026-01-14,spx-up-or-down-on-january-14-2026,2026-01-13 13:00:57.517983+00:00,2026-01-14 21:00:00+00:00,0.0,1.0,False,NaN
9,2026-01-15,spx-up-or-down-on-january-15-2026,2026-01-14 13:01:12.846954+00:00,2026-01-15 21:00:00+00:00,1.0,0.0,False,NaN


In [5]:
resolved_df = df[df["resolved"] == True].copy()
up_days   = (resolved_df["outcome"] == "Up").sum()
down_days = (resolved_df["outcome"] == "Down").sum()

print(f"Resolved markets : {len(resolved_df)}")
print(f"  Up days        : {up_days}  ({up_days/len(resolved_df)*100:.1f}%)")
print(f"  Down days      : {down_days}  ({down_days/len(resolved_df)*100:.1f}%)")
resolved_df[["date", "p_up", "p_down", "outcome"]].tail(10)

Resolved markets : 0
  Up days        : 0  (nan%)
  Down days      : 0  (nan%)


C:\Users\trant\AppData\Local\Temp\ipykernel_28460\4090042016.py:6: RuntimeWarning: invalid value encountered in scalar divide
  print(f"  Up days        : {up_days}  ({up_days/len(resolved_df)*100:.1f}%)")
C:\Users\trant\AppData\Local\Temp\ipykernel_28460\4090042016.py:7: RuntimeWarning: invalid value encountered in scalar divide
  print(f"  Down days      : {down_days}  ({down_days/len(resolved_df)*100:.1f}%)")


,date,p_up,p_down,outcome


In [6]:
parquet_path = OUTPUT.replace(".csv", ".parquet")
df.to_parquet(parquet_path, index=False)
print(f"Saved Parquet to {parquet_path}")

Saved Parquet to data_processed/spx_daily_history.parquet
